# LeetCode 4055 — Count Shadow Pairs II (Hard)

Bài toán LeetCode:

```python
class Solution:
    def shadowPairs(self, nums: List[int]) -> int:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4055 (Weekly Contest 519 — câu 4, rating khoảng 2630).

Đây là một bài **rất khó**, nhưng có thể đi lên từng bậc rất tự nhiên:

1. Brute Force `O(n³)` — đọc thẳng định nghĩa.
2. `O(n²)` — với mỗi `i`, quét `j` sang phải và giữ **một con số** duy nhất.
3. `O(n log n)` — **chia để trị theo giá trị** (không phải theo vị trí!) kết hợp **hai ngăn xếp đơn điệu**.

---

## Sau notebook này, bạn sẽ nắm được

1. Định nghĩa cặp bóng (shadow pair) và cách kiểm tra.
2. Kỹ thuật "giữ min đang chạy" để bỏ một vòng lặp.
3. Ý tưởng chia để trị **trên miền giá trị**: tách giá trị nhỏ / lớn, bài con độc lập.
4. Đếm cặp chéo bằng ngăn xếp đơn điệu + "ảnh chụp" (snapshot) kích thước ngăn xếp.
5. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho mảng `nums`. Cặp chỉ số `(i, j)` là **cặp bóng** nếu:

- `i < j`;
- `nums[i] < nums[j]`;
- **không** tồn tại `k` với `i < k < j` mà `nums[i] < nums[k] < nums[j]` (không có phần tử nào ở giữa "chen" vào khoảng giá trị **mở** `(nums[i], nums[j])`).

Đếm số cặp bóng.

Hình dung: đứng ở `i` nhìn sang phải tới `j`; cặp hợp lệ khi không có giá trị nào ở giữa nằm **lọt khe** giữa hai đầu mút.

---

## Ví dụ 1

```python
nums = [3, 1, 4, 2, 5]   ->   5
```

Các cặp bóng (theo chỉ số): `(0,2)` 3→4, `(1,2)` 1→4, `(1,3)` 1→2, `(2,4)` 4→5, `(3,4)` 2→5.

Ví dụ cặp **không** hợp lệ: `(0,4)` 3→5 vì ở giữa có `4 ∈ (3, 5)`; `(1,4)` 1→5 vì có `4` (và `2`).

## Ví dụ 2

```python
nums = [6, 7, 8, 9]   ->   3
```

Chỉ các cặp kề nhau `(0,1), (1,2), (2,3)`; cặp `(0,2)` bị `7` chen giữa.

---

## Ràng buộc

```text
3 <= nums.length <= 5 * 10^4
1 <= nums[i] <= 10^9
```

# 2. Kiến thức nền tảng

## 2.1. Khoảng mở và bằng nhau

Điều kiện chặn là `nums[i] < nums[k] < nums[j]` — **chặt cả hai phía**. Phần tử bằng `nums[i]` hoặc bằng `nums[j]` **không** chặn.

In [1]:
from typing import List

nums = [2, 2, 5, 5]
# cặp (0, 3): 2 -> 5, ở giữa có 2 và 5 — không giá trị nào nằm TRONG (2, 5)
print(any(2 < nums[k] < 5 for k in range(1, 3)))

False


## 2.2. Ngăn xếp đơn điệu (monotonic stack)

Ngăn xếp không tăng: khi đẩy `v`, bỏ các phần tử ở đỉnh **nhỏ hơn** `v`. Phần tử còn lại là những phần tử "chưa bị phần tử nào lớn hơn đứng sau che mất".

In [2]:
stack = []
for v in [3, 1, 4, 2, 2]:
    while stack and stack[-1] < v:
        stack.pop()
    stack.append(v)
    print(f"đẩy {v}: {stack}")

đẩy 3: [3]
đẩy 1: [3, 1]
đẩy 4: [4]
đẩy 2: [4, 2]
đẩy 2: [4, 2, 2]


## 2.3. Gom chỉ số theo giá trị

`groups[r]` = danh sách chỉ số (tăng dần) của giá trị nhỏ thứ `r`.

In [3]:
nums = [3, 1, 4, 2, 5, 1]
vals = sorted(set(nums))
groups = [[i for i, x in enumerate(nums) if x == v] for v in vals]
print(vals)
print(groups)

[1, 2, 3, 4, 5]
[[1, 5], [3], [0], [2], [4]]


# 3. Cách 1 — Brute Force `O(n³)`

Thử mọi cặp, kiểm tra mọi `k` ở giữa.

In [4]:
def shadow_pairs_bruteforce(nums: List[int]) -> int:
    n = len(nums)
    count = 0
    for i in range(n):
        for j in range(i + 1, n):
            if nums[i] < nums[j] and not any(nums[i] < nums[k] < nums[j] for k in range(i + 1, j)):
                count += 1
    return count


print(shadow_pairs_bruteforce([3, 1, 4, 2, 5]), shadow_pairs_bruteforce([6, 7, 8, 9]))

5 3


# 4. Cách 2 — `O(n²)`: giữ "min các giá trị lớn hơn `nums[i]`"

Cố định `i`, quét `j = i+1, i+2, ...`. Đặt

```text
m = min { nums[k] : i < k < j, nums[k] > nums[i] }      (m = +∞ nếu chưa có)
```

Cặp `(i, j)` hợp lệ ⇔ `nums[j] > nums[i]` **và** không có `nums[k]` ở giữa nằm trong `(nums[i], nums[j])` ⇔ `nums[j] <= m`.

(Vì mọi `nums[k] > nums[i]` đều `>= m`; điều kiện "không ai `< nums[j]`" tương đương "phần tử nhỏ nhất trong số đó `>= nums[j]`".)

Sau khi xét `j`, cập nhật `m = min(m, nums[j])` nếu `nums[j] > nums[i]`.

## Mô phỏng `i = 1` (`nums[i] = 1`) với `nums = [3, 1, 4, 2, 5]`

| j | nums[j] | `> 1`? | m trước | `nums[j] <= m`? | đếm | m sau |
|---:|---:|:---:|---:|:---:|---:|---:|
| 2 | 4 | có | ∞ | có | 1 | 4 |
| 3 | 2 | có | 4 | có | 2 | 2 |
| 4 | 5 | có | 2 | không | 2 | 2 |

In [5]:
def shadow_pairs_n2(nums: List[int], trace_i=None) -> int:
    n = len(nums)
    count = 0
    for i in range(n):
        m = float("inf")
        for j in range(i + 1, n):
            if nums[j] > nums[i]:
                ok = nums[j] <= m
                if ok:
                    count += 1
                if i == trace_i:
                    print(f"i={i} j={j} nums[j]={nums[j]} m_trước={m} hợp lệ={ok}")
                m = min(m, nums[j])
    return count


print(shadow_pairs_n2([3, 1, 4, 2, 5], trace_i=1))

i=1 j=2 nums[j]=4 m_trước=inf hợp lệ=True
i=1 j=3 nums[j]=2 m_trước=4 hợp lệ=True
i=1 j=4 nums[j]=5 m_trước=2 hợp lệ=False
5


`n = 5·10^4` → `n² / 2 ≈ 1.25·10^9` bước — quá chậm với Python. Cần ý tưởng mới.

---

# 5. Cách 3 — Chia để trị trên **miền giá trị**

## 5.1. Quan sát then chốt

Chia các **giá trị** thành nửa nhỏ `Low` và nửa lớn `High` (theo thứ hạng). Xét cặp `(i, j)`:

- **Cả hai thuộc `Low`:** phần tử chặn phải có giá trị nằm **giữa** `nums[i]` và `nums[j]`, nên cũng thuộc `Low`. → Bài con chỉ cần **dãy con các phần tử thuộc `Low`** (giữ nguyên thứ tự)! Phần tử `High` có thể xóa đi mà không ảnh hưởng.
- **Cả hai thuộc `High`:** tương tự, bài con độc lập trên dãy con `High`.
- **Cặp chéo:** `nums[i] ∈ Low`, `nums[j] ∈ High` (tự động `nums[i] < nums[j]`) và `i < j`. Đây là phần cần đếm khi "trộn".

Đệ quy theo giá trị: mỗi tầng tổng độ dài các dãy con là `n`, có `log(số giá trị khác nhau)` tầng → nếu trộn được trong thời gian tuyến tính thì tổng `O(n log n)`.

Ta cài đặt **từ dưới lên**: mỗi giá trị khác nhau là một nhóm (danh sách chỉ số); lần lượt trộn từng cặp nhóm kề nhau về giá trị, mỗi lần đếm cặp chéo.

## 5.2. Đếm cặp chéo — ai có thể chặn?

Cặp chéo `(i, j)` với `i ∈ Low`, `j ∈ High`, `i < j` bị chặn bởi `k` (vị trí giữa `i` và `j`) khi:

1. **`k ∈ Low` và `nums[k] > nums[i]`** (luôn `< nums[j]`). → `i` hợp lệ chỉ khi **không** có phần tử `Low` nào **lớn hơn** đứng sau nó (tới `j`). Các `i` như vậy chính là phần tử còn lại trong **ngăn xếp không tăng** `s1` của các giá trị `Low` đã gặp!
2. **`k ∈ High` và `nums[k] < nums[j]`** (luôn `> nums[i]`). → gọi `p` là vị trí phần tử `High` **gần nhất bên trái `j`** có giá trị `< nums[j]`. Khi đó `i` phải nằm **sau** `p`.

Vậy số cặp cho `j` = số phần tử trong `s1` đứng **sau** `p`.

## 5.3. Cài đặt bằng snapshot

- `s1`: ngăn xếp không tăng các giá trị `Low` (theo thứ tự vị trí).
- `s2`: ngăn xếp các phần tử `High` đã gặp, giá trị **tăng ngặt** từ đáy lên; khi gặp `j`, bỏ các đỉnh có giá trị `>= nums[j]` → đỉnh còn lại chính là `p`.
- Mỗi phần tử trong `s2` lưu kèm `snap` = **số phần tử của `s1` nằm trước nó còn sống**. Vì `s1` là ngăn xếp, các phần tử trước `p` còn sống là **một tiền tố** của `s1` dài `snap`.
- Khi `s1` bị pop, tiền tố có thể ngắn đi: cập nhật `snap` của đỉnh `s2` thành `min(snap, len(s1))`. Khi pop `s2`, chuyển `min` xuống phần tử mới ở đỉnh (vì nó nằm trước, tiền tố của nó không thể dài hơn).
- Đếm cho `j`: `len(s1) - (snap của đỉnh s2, hoặc 0 nếu s2 rỗng)`.

In [6]:
def merge_and_count(nums, left, right, verbose=False):
    # left: chỉ số các phần tử Low (tăng dần), right: chỉ số các phần tử High (tăng dần)
    merged, cnt = [], 0
    s1, s2 = [], []          # s1: giá trị Low (không tăng); s2: [giá trị High, snap]
    i = 0
    for idx in right:
        while i < len(left) and left[i] < idx:        # đưa các phần tử Low đứng trước idx vào s1
            v = nums[left[i]]
            while s1 and s1[-1] < v:
                s1.pop()
            if s2:
                s2[-1][1] = min(s2[-1][1], len(s1))
            s1.append(v)
            merged.append(left[i])
            i += 1
        while s2 and s2[-1][0] >= nums[idx]:          # tìm p: High gần nhất bên trái có giá trị < nums[idx]
            base = s2.pop()[1]
            if s2:
                s2[-1][1] = min(s2[-1][1], base)
        add = len(s1) - (s2[-1][1] if s2 else 0)
        if verbose:
            print(f"  j={idx} (giá trị {nums[idx]}): s1={s1}, p={'không có' if not s2 else s2[-1]}, +{add}")
        cnt += add
        s2.append([nums[idx], len(s1)])
        merged.append(idx)
    merged += left[i:]
    return merged, cnt

In [7]:
def shadow_pairs_dc(nums: List[int], verbose=False) -> int:
    vals = sorted(set(nums))
    rank = {v: r for r, v in enumerate(vals)}
    groups = [[] for _ in vals]
    for i, x in enumerate(nums):
        groups[rank[x]].append(i)
    total = 0
    level = 0
    while len(groups) > 1:
        level += 1
        new_groups = []
        for g in range(0, len(groups) - 1, 2):
            if verbose:
                print(f"Tầng {level}: trộn Low={[nums[t] for t in groups[g]]} (vị trí {groups[g]}) "
                      f"với High={[nums[t] for t in groups[g + 1]]} (vị trí {groups[g + 1]})")
            merged, c = merge_and_count(nums, groups[g], groups[g + 1], verbose)
            total += c
            new_groups.append(merged)
        if len(groups) % 2:
            new_groups.append(groups[-1])
        groups = new_groups
    return total


print(shadow_pairs_dc([3, 1, 4, 2, 5], verbose=True))

Tầng 1: trộn Low=[1] (vị trí [1]) với High=[2] (vị trí [3])
  j=3 (giá trị 2): s1=[1], p=không có, +1
Tầng 1: trộn Low=[3] (vị trí [0]) với High=[4] (vị trí [2])
  j=2 (giá trị 4): s1=[3], p=không có, +1
Tầng 2: trộn Low=[1, 2] (vị trí [1, 3]) với High=[3, 4] (vị trí [0, 2])
  j=0 (giá trị 3): s1=[], p=không có, +0
  j=2 (giá trị 4): s1=[1], p=[3, 0], +1
Tầng 3: trộn Low=[3, 1, 4, 2] (vị trí [0, 1, 2, 3]) với High=[5] (vị trí [4])
  j=4 (giá trị 5): s1=[4, 2], p=không có, +2
5


## 5.4. Đọc lại vết chạy `nums = [3, 1, 4, 2, 5]`

| tầng | Low (giá trị) | High (giá trị) | cặp chéo tìm được | + |
|---:|---|---|---|---:|
| 1 | `{1}` | `{2}` | `(1,3)` 1→2 | 1 |
| 1 | `{3}` | `{4}` | `(0,2)` 3→4 — giá trị `1` ở giữa **không thuộc** bài con này | 1 |
| 2 | `{1,2}` | `{3,4}` | `(1,2)` 1→4 | 1 |
| 3 | `{1,2,3,4}` | `{5}` | `(2,4)` 4→5, `(3,4)` 2→5; `s1 = [4, 2]` (3 và 1 bị 4 che) | 2 |

Tổng `1 + 1 + 1 + 2 = 5` ✔. Ở tầng 3, `3` và `1` đã bị pop khỏi `s1` vì `4` lớn hơn đứng sau — đúng là `4` chặn cặp `3→5` và `1→5`.

## 5.5. Một ví dụ cho thấy vai trò của `s2`

`nums = [2, 1, 4, 5]`. Cặp hợp lệ: `2→4` (ở giữa có `1` không thuộc `(2, 4)`), `1→4` (kề), `4→5` (kề) — tổng **3**. Cặp `2→5` và `1→5` đều bị `4` chặn — đây là phần tử **`High`** chặn, trong khi `s1` không hề pop gì.

(Trong vết chạy, `p` được in dạng `[giá trị, snap]`.)

In [8]:
print(shadow_pairs_dc([2, 1, 4, 5], verbose=True), shadow_pairs_bruteforce([2, 1, 4, 5]))

Tầng 1: trộn Low=[1] (vị trí [1]) với High=[2] (vị trí [0])
  j=0 (giá trị 2): s1=[], p=không có, +0
Tầng 1: trộn Low=[4] (vị trí [2]) với High=[5] (vị trí [3])
  j=3 (giá trị 5): s1=[4], p=không có, +1
Tầng 2: trộn Low=[2, 1] (vị trí [0, 1]) với High=[4, 5] (vị trí [2, 3])
  j=2 (giá trị 4): s1=[2, 1], p=không có, +2
  j=3 (giá trị 5): s1=[2, 1], p=[4, 2], +0
3 3


Ở tầng 2 (`Low = {1, 2}`, `High = {4, 5}`):

- `j` ứng với `4` (vị trí 2): `s1 = [2, 1]` (không tăng, cả hai còn sống), `s2` rỗng → +2 (`2→4`, `1→4`). Đẩy `[4, snap = 2]` vào `s2`.
- `j` ứng với `5` (vị trí 3): đỉnh `s2` là `4 < 5` nên giữ lại → `p` là phần tử `4` với `snap = 2`: **cả hai** phần tử của `s1` nằm trước nó → `+ (2 - 2) = 0`.

Nếu bỏ `s2`, ta sẽ đếm nhầm thêm 2 cặp (xem Lỗi 2).

## 5.6. Ví dụ chặn từ cả hai phía: `[1, 3, 2, 5]`

In [9]:
print(shadow_pairs_dc([1, 3, 2, 5], verbose=True), shadow_pairs_bruteforce([1, 3, 2, 5]))

Tầng 1: trộn Low=[1] (vị trí [0]) với High=[2] (vị trí [2])
  j=2 (giá trị 2): s1=[1], p=không có, +1
Tầng 1: trộn Low=[3] (vị trí [1]) với High=[5] (vị trí [3])
  j=3 (giá trị 5): s1=[3], p=không có, +1
Tầng 2: trộn Low=[1, 2] (vị trí [0, 2]) với High=[3, 5] (vị trí [1, 3])
  j=1 (giá trị 3): s1=[1], p=không có, +1
  j=3 (giá trị 5): s1=[2], p=[3, 0], +1
4 4


Ở tầng 2 (`Low = {1, 2}`, `High = {3, 5}`), khi `2` (vị trí 2) vào `s1`, nó pop `1` (vì `2` chặn `1→5` từ phía `Low`), và snapshot của phần tử `3` trong `s2` giảm từ 1 xuống `min(1, 0) = 0`. Nhờ cập nhật đó, với `j` ứng với `5` ta đếm `1 - 0 = 1` (cặp `2→5`) chứ không phải `1 - 1 = 0`. Cặp `1→3` được đếm trước đó khi `j` ứng với `3`.

# 6. Phiên bản đúng format LeetCode

In [10]:
from typing import List


class Solution:
    def shadowPairs(self, nums: List[int]) -> int:
        vals = sorted(set(nums))
        rank = {v: r for r, v in enumerate(vals)}
        groups = [[] for _ in vals]
        for i, x in enumerate(nums):
            groups[rank[x]].append(i)

        def merge(left, right):
            merged, cnt, s1, s2, i = [], 0, [], [], 0
            for idx in right:
                while i < len(left) and left[i] < idx:
                    v = nums[left[i]]
                    while s1 and s1[-1] < v:
                        s1.pop()
                    if s2:
                        s2[-1][1] = min(s2[-1][1], len(s1))
                    s1.append(v)
                    merged.append(left[i])
                    i += 1
                while s2 and s2[-1][0] >= nums[idx]:
                    base = s2.pop()[1]
                    if s2:
                        s2[-1][1] = min(s2[-1][1], base)
                cnt += len(s1) - (s2[-1][1] if s2 else 0)
                s2.append([nums[idx], len(s1)])
                merged.append(idx)
            merged += left[i:]
            return merged, cnt

        total = 0
        while len(groups) > 1:
            nxt = []
            for g in range(0, len(groups) - 1, 2):
                merged, c = merge(groups[g], groups[g + 1])
                total += c
                nxt.append(merged)
            if len(groups) % 2:
                nxt.append(groups[-1])
            groups = nxt
        return total

In [11]:
sol = Solution()
tests = [
    ([3, 1, 4, 2, 5], 5, "ví dụ 1"),
    ([6, 7, 8, 9], 3, "ví dụ 2"),
    ([5, 5, 5], 0, "toàn bằng nhau"),
    ([9, 8, 7, 6], 0, "giảm dần"),
    ([2, 1, 4, 5], 3, "High chặn (vai trò s2)"),
    ([1, 3, 2, 5], 4, "chặn từ hai phía"),
    ([2, 2, 5, 5], 4, "bằng nhau không chặn"),
]
for nums, expected, name in tests:
    result = sol.shadowPairs(nums)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:22} -> {result}")

PASS | ví dụ 1                -> 5
PASS | ví dụ 2                -> 3
PASS | toàn bằng nhau         -> 0
PASS | giảm dần               -> 0
PASS | High chặn (vai trò s2) -> 3
PASS | chặn từ hai phía       -> 4
PASS | bằng nhau không chặn   -> 4


Test `[2, 2, 5, 5]`: mọi cặp `2→5` (bốn cặp) đều hợp lệ vì ở giữa chỉ có `2` hoặc `5` — không nằm **trong** khoảng mở `(2, 5)`.

# 7. Vì sao đúng?

1. **Độc lập của bài con:** phần tử chặn cặp `(i, j)` có giá trị nằm giữa `nums[i]` và `nums[j]`. Nếu cả hai cùng nửa giá trị thì phần tử chặn cũng cùng nửa — xóa nửa kia không đổi kết quả.
2. **Mỗi cặp được đếm đúng một lần:** cặp `(i, j)` với `nums[i] < nums[j]` được đếm ở đúng lần trộn đầu tiên mà hai giá trị rơi vào hai nhóm khác nhau (`nums[i]` bên `Low`, `nums[j]` bên `High`).
3. **Cặp chéo:** chặn từ `Low` ⇔ có phần tử `Low` lớn hơn `nums[i]` đứng sau `i` (trước `j`) ⇔ `i` đã bị pop khỏi `s1`. Chặn từ `High` ⇔ có phần tử `High` giá trị `< nums[j]` đứng sau `i` ⇔ `i` đứng trước `p` (`p` là phần tử như vậy gần `j` nhất). Snapshot đếm chính xác các phần tử `s1` còn sống đứng trước `p`.

# 8. Độ phức tạp

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | thử mọi cặp + mọi k | `O(n³)` | `O(1)` |
| 2. Min đang chạy | cố định i, quét j | `O(n²)` | `O(1)` |
| 3. Chia để trị theo giá trị | trộn nhóm + 2 ngăn xếp | `O(n log n)` | `O(n)` |

Mỗi tầng trộn tổng cộng `O(n)` (mỗi chỉ số vào/ra mỗi ngăn xếp tối đa một lần), có `⌈log₂(số giá trị khác nhau)⌉` tầng.

In [12]:
import time
import random

random.seed(4055)
big = [random.randint(1, 10**9) for _ in range(5 * 10**4)]
start = time.time()
sol.shadowPairs(big)
print("n = 50000:", round(time.time() - start, 2), "giây")

n = 50000: 0.33 giây


# 9. Đối chiếu ba cách

In [13]:
random.seed(55)
for _ in range(3000):
    nums = [random.randint(1, 6) for _ in range(random.randint(3, 10))]
    expected = shadow_pairs_bruteforce(nums)
    assert shadow_pairs_n2(nums) == expected, nums
    assert shadow_pairs_dc(nums) == expected, nums
    assert sol.shadowPairs(nums) == expected, nums
for _ in range(200):
    nums = [random.randint(1, 50) for _ in range(random.randint(3, 120))]
    assert shadow_pairs_n2(nums) == sol.shadowPairs(nums), nums
print("Cả ba cách khớp nhau trên 3200 test")

Cả ba cách khớp nhau trên 3200 test


# 10. Những lỗi dễ mắc

## Lỗi 1 — Dùng `<=` khi pop `s1` (coi phần tử bằng là "che")

Phần tử `Low` **bằng** `nums[i]` không chặn (khoảng mở). Nếu pop cả phần tử bằng, ta mất các cặp như `2→5` đầu tiên trong `[2, 2, 5]`.

In [14]:
def wrong_pop_equal(nums):
    vals = sorted(set(nums)); rank = {v: r for r, v in enumerate(vals)}
    groups = [[] for _ in vals]
    for i, x in enumerate(nums):
        groups[rank[x]].append(i)

    def merge(left, right):
        merged, cnt, s1, s2, i = [], 0, [], [], 0
        for idx in right:
            while i < len(left) and left[i] < idx:
                v = nums[left[i]]
                while s1 and s1[-1] <= v:          # SAI: pop cả phần tử bằng
                    s1.pop()
                if s2:
                    s2[-1][1] = min(s2[-1][1], len(s1))
                s1.append(v); merged.append(left[i]); i += 1
            while s2 and s2[-1][0] >= nums[idx]:
                base = s2.pop()[1]
                if s2:
                    s2[-1][1] = min(s2[-1][1], base)
            cnt += len(s1) - (s2[-1][1] if s2 else 0)
            s2.append([nums[idx], len(s1)]); merged.append(idx)
        return merged + left[i:], cnt

    total = 0
    while len(groups) > 1:
        nxt = []
        for g in range(0, len(groups) - 1, 2):
            m, c = merge(groups[g], groups[g + 1]); total += c; nxt.append(m)
        if len(groups) % 2:
            nxt.append(groups[-1])
        groups = nxt
    return total


print("Sai :", wrong_pop_equal([2, 2, 5]), "| Đúng:", sol.shadowPairs([2, 2, 5]))

Sai : 1 | Đúng: 2


## Lỗi 2 — Quên chặn từ phía `High` (bỏ `s2`)

Chỉ dùng `s1` sẽ đếm cả cặp bị một phần tử `High` nhỏ hơn `nums[j]` chen giữa.

In [15]:
def wrong_no_s2_cross(nums, left, right):
    cnt, s1, i = 0, [], 0
    for idx in right:
        while i < len(left) and left[i] < idx:
            v = nums[left[i]]
            while s1 and s1[-1] < v:
                s1.pop()
            s1.append(v); i += 1
        cnt += len(s1)                                   # SAI: không trừ phần bị High chặn
    return cnt


nums = [2, 1, 4, 5]
print("Cặp chéo Low={1,2} / High={4,5}: sai =", wrong_no_s2_cross(nums, [0, 1], [2, 3]),
      "| đúng =", merge_and_count(nums, [0, 1], [2, 3])[1])

Cặp chéo Low={1,2} / High={4,5}: sai = 4 | đúng = 2


## Lỗi 3 — Quên cập nhật snapshot khi `s1` bị pop

Nếu không `min(snap, len(s1))`, snapshot có thể lớn hơn kích thước thật của tiền tố → trừ quá tay, thậm chí ra số âm.

In [16]:
def wrong_no_snapshot_update(nums, left, right):
    cnt, s1, s2, i = 0, [], [], 0
    for idx in right:
        while i < len(left) and left[i] < idx:
            v = nums[left[i]]
            while s1 and s1[-1] < v:
                s1.pop()
            s1.append(v); i += 1                          # SAI: không cập nhật s2[-1][1]
        while s2 and s2[-1][0] >= nums[idx]:
            s2.pop()
        cnt += len(s1) - (s2[-1][1] if s2 else 0)
        s2.append([nums[idx], len(s1)])
    return cnt


random.seed(3)
for _ in range(5000):
    nums = [random.randint(1, 6) for _ in range(random.randint(3, 8))]
    vals = sorted(set(nums))
    if len(vals) < 2:
        continue
    mid = vals[len(vals) // 2]
    left = [i for i, x in enumerate(nums) if x < mid]
    right = [i for i, x in enumerate(nums) if x >= mid]
    good = merge_and_count(nums, left, right)[1]
    bad = wrong_no_snapshot_update(nums, left, right)
    if good != bad:
        print("Phản ví dụ:", nums, "Low:", [nums[t] for t in left], "| sai:", bad, "| đúng:", good)
        break

Phản ví dụ: [2, 4, 5, 6, 4, 3, 3, 5] Low: [2, 3, 3] | sai: 3 | đúng: 4


## Lỗi 4 — Chia để trị theo **vị trí** (như merge sort thường)

Chia mảng làm hai nửa trái/phải theo chỉ số: cặp chéo `(i ∈ trái, j ∈ phải)` có thể bị chặn bởi phần tử ở **cả hai nửa** với điều kiện phụ thuộc cả `nums[i]` lẫn `nums[j]` — khó đếm. Chia theo **giá trị** làm điều kiện tách thành hai phần đơn giản (một chỉ phụ thuộc `i`, một chỉ phụ thuộc `j`).

## Lỗi 5 — Dùng `O(n²)` cho `n = 5·10^4`

Khoảng `1.25·10^9` phép so sánh — vượt thời gian.

# 11. Mẹo nhận diện

1. Điều kiện "**không có phần tử ở giữa nằm trong khoảng giá trị** giữa hai đầu mút" → nghĩ tới **chia để trị theo giá trị**: phần tử chặn có giá trị trung gian nên cùng nửa với cặp.
2. Đếm cặp `(i, j)` với điều kiện "không ai lớn hơn `a[i]` ở giữa" → **ngăn xếp không tăng**.
3. "Không ai nhỏ hơn `a[j]` ở giữa" → phần tử nhỏ hơn gần nhất bên trái → **ngăn xếp tăng**.
4. Kết hợp hai điều kiện trên hai ngăn xếp → lưu **snapshot kích thước** để biết "phần còn sống trước vị trí p".

Các bài liên quan: *Number of Visible People in a Queue* (LeetCode 1944), *Count of Smaller Numbers After Self* (315 — chia để trị), *Sum of Subarray Minimums* (907 — ngăn xếp đơn điệu).

# 12. Biến thể

## Biến thể 1 — Cặp bóng giảm (`nums[i] > nums[j]`)

Đổi dấu toàn bộ mảng: `nums[i] > nums[j]` ⇔ `-nums[i] < -nums[j]`, và khoảng chặn cũng đổi dấu tương ứng. (Bài tập 6.)

## Biến thể 2 — Liệt kê các cặp

Cách 2 liệt kê được ngay trong `O(n²)`. Không thể làm tốt hơn nhiều, vì số cặp bóng có thể lên tới `Θ(n²)`: với mảng xen kẽ `[1, 2, 1, 2, ...]`, **mọi** cặp `1 → 2` (1 đứng trước 2) đều hợp lệ, vì ở giữa chỉ có `1` hoặc `2` — không nằm trong khoảng mở `(1, 2)`. Đó cũng là lý do bài **đếm** cần thuật toán không liệt kê từng cặp.

In [17]:
alt = [1, 2] * 6
print(len(alt), shadow_pairs_bruteforce(alt))

12 21


# 13. Template tư duy

```python
# Chia để trị trên miền giá trị (từ dưới lên)
groups = [chỉ số của từng giá trị khác nhau, theo thứ tự giá trị]
while len(groups) > 1:
    trộn từng cặp (Low, High) kề nhau:
        duyệt theo vị trí, đếm cặp chéo bằng cấu trúc phụ (ngăn xếp / BIT)
```

# 14. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Đếm cặp bóng của `nums = [2, 4, 1, 3]` bằng tay và liệt kê chúng. Kiểm tra bằng cell dưới.

In [18]:
# Kiểm tra Bài 1
nums = [2, 4, 1, 3]
pairs = [(i, j) for i in range(4) for j in range(i + 1, 4)
         if nums[i] < nums[j] and not any(nums[i] < nums[k] < nums[j] for k in range(i + 1, j))]
print(pairs, len(pairs))

[(0, 1), (0, 3), (2, 3)] 3


## Bài 2 — Dự đoán output

```python
print(shadow_pairs_bruteforce([1, 2, 3, 4, 5]))
print(shadow_pairs_bruteforce([1, 5, 1, 5]))
```

In [19]:
print(shadow_pairs_bruteforce([1, 2, 3, 4, 5]))
print(shadow_pairs_bruteforce([1, 5, 1, 5]))

4
3


## Bài 3 — Hoàn thiện Cách 2 (`O(n²)`)

In [20]:
def exercise_3_shadow_n2(nums: List[int]) -> int:
    # TODO: với mỗi i, quét j và giữ m = min các giá trị > nums[i] đã gặp
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_shadow_n2([3, 1, 4, 2, 5]) == 5

## Bài 4 — Phần tử lớn hơn tiếp theo (Next Greater Element)

Với mỗi `i`, trả về chỉ số `j > i` nhỏ nhất có `nums[j] > nums[i]`, hoặc `-1`. Dùng ngăn xếp đơn điệu, `O(n)`.

```python
next_greater([2, 1, 3, 2, 4])   -> [2, 2, 4, 4, -1]
```

In [21]:
def exercise_4_next_greater(nums: List[int]) -> List[int]:
    # TODO
    raise NotImplementedError


# assert exercise_4_next_greater([2, 1, 3, 2, 4]) == [2, 2, 4, 4, -1]

## Bài 5 — Cặp "nhìn thấy nhau"

Đếm cặp `(i, j)`, `i < j`, sao cho mọi phần tử ở giữa đều **nhỏ hơn** `min(nums[i], nums[j])` (giả sử các phần tử đôi một khác nhau). Làm `O(n)` bằng ngăn xếp.

```python
visible_pairs([3, 1, 2, 5])   -> 5    # (0,1) (1,2) (0,2) (2,3) (0,3)
```

In [22]:
def exercise_5_visible_pairs(nums: List[int]) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_visible_pairs([3, 1, 2, 5]) == 5

## Bài 6 — Mini challenge: cặp bóng giảm

Đếm cặp `(i, j)`, `i < j`, `nums[i] > nums[j]`, không có `k` ở giữa với `nums[j] < nums[k] < nums[i]`. Yêu cầu `O(n log n)` (gợi ý: tái sử dụng `Solution`).

In [23]:
def exercise_6_shadow_desc(nums: List[int]) -> int:
    # TODO
    raise NotImplementedError

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: `m = inf` mỗi khi đổi `i`.
- Mức 2: khi `nums[j] > nums[i]`: đếm nếu `nums[j] <= m`.
- Mức 3: luôn cập nhật `m = min(m, nums[j])` trong trường hợp đó.

**Bài 4**

- Mức 1: ngăn xếp lưu **chỉ số** chưa tìm được phần tử lớn hơn.
- Mức 2: gặp `j`: pop mọi `i` có `nums[i] < nums[j]`, gán `ans[i] = j`.
- Mức 3: đẩy `j` vào.

**Bài 5**

- Mức 1: ngăn xếp giảm dần.
- Mức 2: gặp `x`: mỗi phần tử bị pop (nhỏ hơn `x`) tạo một cặp với `x`.
- Mức 3: nếu ngăn xếp còn phần tử sau khi pop, đỉnh cũng thấy `x` → +1.

**Bài 6**

- Mức 1: nhân `-1` cho mọi phần tử.
- Mức 2: `nums[i] > nums[j]` ⇔ `-nums[i] < -nums[j]`.
- Mức 3: `nums[j] < nums[k] < nums[i]` ⇔ `-nums[i] < -nums[k] < -nums[j]`.

# 15. Đáp án tham khảo

## Đáp án Bài 1

`nums = [2, 4, 1, 3]`:

- `(0,1)` 2→4: không có gì ở giữa ✔
- `(0,3)` 2→3: ở giữa `4, 1` — không giá trị nào trong `(2, 3)` ✔
- `(2,3)` 1→3: kề nhau ✔
- `(1, ·)`: không có giá trị nào lớn hơn 4 phía sau.

Tổng **3** cặp.

## Đáp án Bài 2

```text
4
3
```

`[1, 2, 3, 4, 5]`: chỉ các cặp kề nhau (4 cặp). `[1, 5, 1, 5]`: `(0,1)`, `(2,3)`, và `(0,3)` 1→5 (ở giữa chỉ có `5`, `1` — không nằm trong `(1, 5)`); còn `(0,2)` và `(1,3)` có hai giá trị bằng nhau nên không phải cặp. Tổng 3.

In [24]:
# Đáp án Bài 3
def answer_3_shadow_n2(nums: List[int]) -> int:
    n, count = len(nums), 0
    for i in range(n):
        m = float("inf")
        for j in range(i + 1, n):
            if nums[j] > nums[i]:
                if nums[j] <= m:
                    count += 1
                m = min(m, nums[j])
    return count


assert answer_3_shadow_n2([3, 1, 4, 2, 5]) == 5
print("Bài 3: PASS")

Bài 3: PASS


In [25]:
# Đáp án Bài 4
def answer_4_next_greater(nums: List[int]) -> List[int]:
    ans = [-1] * len(nums)
    stack = []
    for j, x in enumerate(nums):
        while stack and nums[stack[-1]] < x:
            ans[stack.pop()] = j
        stack.append(j)
    return ans


assert answer_4_next_greater([2, 1, 3, 2, 4]) == [2, 2, 4, 4, -1]
print("Bài 4: PASS")

Bài 4: PASS


In [26]:
# Đáp án Bài 5
def answer_5_visible_pairs(nums: List[int]) -> int:
    stack, count = [], 0
    for x in nums:
        while stack and stack[-1] < x:
            stack.pop()
            count += 1
        if stack:
            count += 1
        stack.append(x)
    return count


def brute_visible(nums):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n)
               if all(nums[k] < min(nums[i], nums[j]) for k in range(i + 1, j)))


assert answer_5_visible_pairs([3, 1, 2, 5]) == 5
random.seed(5)
for _ in range(500):
    arr = random.sample(range(1, 30), random.randint(1, 9))
    assert answer_5_visible_pairs(arr) == brute_visible(arr), arr
print("Bài 5: PASS")

Bài 5: PASS


In [27]:
# Đáp án Bài 6
def answer_6_shadow_desc(nums: List[int]) -> int:
    return Solution().shadowPairs([-x for x in nums])


def brute_desc(nums):
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n)
               if nums[i] > nums[j] and not any(nums[j] < nums[k] < nums[i] for k in range(i + 1, j)))


random.seed(6)
for _ in range(1000):
    arr = [random.randint(1, 6) for _ in range(random.randint(3, 9))]
    assert answer_6_shadow_desc(arr) == brute_desc(arr), arr
print("Bài 6: PASS (khớp brute force)")

Bài 6: PASS (khớp brute force)


# 16. Tổng kết

**Count Shadow Pairs II**:

1. Định nghĩa: không có phần tử ở giữa nằm **trong khoảng mở** giá trị của hai đầu mút.
2. `O(n²)`: cố định `i`, giữ `m` = min các giá trị lớn hơn `nums[i]`; hợp lệ ⇔ `nums[i] < nums[j] <= m`.
3. `O(n log n)`: chia để trị **theo giá trị** — bài con độc lập; cặp chéo bị chặn từ `Low` (ngăn xếp không tăng `s1`) hoặc từ `High` (phần tử nhỏ hơn gần nhất trong `s2`); snapshot đếm phần `s1` còn sống trước `p`.

## Độ phức tạp

```text
Time:  O(n log n)
Space: O(n)
```

# 17. Checklist trước khi submit

- [ ] Khoảng chặn là **mở** (phần tử bằng không chặn) — pop `s1` dùng `<`, pop `s2` dùng `>=`?
- [ ] Cập nhật snapshot khi `s1` bị pop và khi `s2` bị pop?
- [ ] Nhóm lẻ cuối được chuyển nguyên sang tầng sau?
- [ ] Đã test: toàn bằng nhau, giảm dần, `[2, 2, 5, 5]`, `[2, 1, 4, 5]`, `[1, 3, 2, 5]`?